In [80]:
using Sunny, CairoMakie, ColorSchemes, Colors, LinearAlgebra, ProgressMeter
# Cairo for static, GL for interactive

include("/Users/fbnielsen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Uni OneDrive/4 år/Clinoatacamite/Sunny_models/func_clino1.jl")
units = Units(:meV, :angstrom)

Units(:meV, :angstrom)

# Final Q-Mag project

## Spin Ice lattice Holmiumtitenate  $(\text{Ho}_2\text{Ti}_2\text{O}_7)$

### Definition of crystal

In [81]:
latvecs = lattice_vectors(7.175, 7.175, 7.175, 90, 90, 90) #https://legacy.materialsproject.org/materials/mp-33948/

positions = [[0.875, 0.625, 0.375],
             [0.625, 0.125, 0.625],
             [0.875, 0.875, 0.125],
             [0.625, 0.875, 0.375],
             [0.875, 0.125, 0.875],
             [0.625, 0.625, 0.125],
             [0.875, 0.375, 0.625],
             [0.625, 0.375, 0.875],
             [0.375, 0.625, 0.875],
             [0.125, 0.125, 0.125],
             [0.375, 0.875, 0.625],
             [0.125, 0.875, 0.875],
             [0.375, 0.125, 0.375],
             [0.125, 0.625, 0.625],
             [0.375, 0.375, 0.125],
             [0.125, 0.375, 0.375]
             ] # From other pyrochlore material: https://sunnysuite.github.io/Sunny.jl/stable/examples/contributed/MgCr2O4-tutorial.html#Setting-up-the-crystal-structure

types = ["Ho" for _ in positions]

cryst = Crystal(latvecs, positions; types=types)  # Fd3m [227]

sys = System(cryst, [1 => Moment(s=8, g=2)] , :dipole) 

sys = reshape_supercell(sys, primitive_cell(cryst))
sys = reshape_supercell(sys, [1 0 0; 0 1 0; 0 0 1])

System [Dipole mode]
Supercell (1×1×1)×16
Energy per site 0


In [101]:
cryst

Crystal
Spacegroup 'F d -3 m' (227)
Lattice params a=7.175, b=7.175, c=7.175, α=90°, β=90°, γ=90°
Cell volume 369.4
Type 'Ho', Wyckoff 16c (site sym. '.-3m'):
   1. [7/8, 5/8, 3/8]
   2. [5/8, 1/8, 5/8]
   3. [7/8, 7/8, 1/8]
   4. [5/8, 7/8, 3/8]
   5. [7/8, 1/8, 7/8]
   6. [5/8, 5/8, 1/8]
   7. [7/8, 3/8, 5/8]
   8. [5/8, 3/8, 7/8]
   9. [3/8, 5/8, 7/8]
   10. [1/8, 1/8, 1/8]
   11. [3/8, 7/8, 5/8]
   12. [1/8, 7/8, 7/8]
   13. [3/8, 1/8, 3/8]
   14. [1/8, 5/8, 5/8]
   15. [3/8, 3/8, 1/8]
   16. [1/8, 3/8, 3/8]


In [82]:
view_crystal(sys)

In [ ]:
J_nn = 0.52 * Sunny.meV_per_K
set_exchange!(sys, J_nn, Bond(1, 3, [0,0,0]))

S = spin_matrices(8)
O = stevens_matrices(spin_label(sys, 1))

D = 5*J_nn # dipolar: 2.35 * Sunny.meV_per_K
set_onsite_coupling!(sys, -D * (-O[2,-2]-2*O[2,-1]+2*O[2,1]), 1) 

sys_lr = clone_system(sys)
# enable_dipole_dipole!(sys_lr, units.vacuum_permeability)

sys_extend = reshape_supercell(sys_lr, [10 0 0; 0 10 0; 0 0 10])
randomize_spins!(sys_extend)
minimize_energy!(sys_extend, maxiters=10000)

display(energy_per_site(sys_extend))

-2.8678485096419144

┌ Warning: Overwriting coupling for Bond(1, 3, [0, 0, 0])
└ @ Sunny /Users/fbnielsen/.julia/packages/Sunny/I2Xia/src/System/ModelParams.jl:19


In [124]:
print_site(cryst, 1)

Atom 1
Type 'Ho', position [7/8, 5/8, 3/8], Wyckoff 16c
Allowed g-tensor: [ A B -B
                    B A  B
                   -B B  A]
Allowed anisotropy in Stevens operators:
    c₁*(-𝒪[2,-2]-2𝒪[2,-1]+2𝒪[2,1]) +
    c₂*(7𝒪[4,-3]+2𝒪[4,-2]-𝒪[4,-1]+𝒪[4,1]+7𝒪[4,3]) + c₃*(𝒪[4,0]+5𝒪[4,4]) +
    c₄*(11𝒪[6,-6]+8𝒪[6,-3]-𝒪[6,-2]+8𝒪[6,-1]-8𝒪[6,1]+8𝒪[6,3]) + c₅*(-𝒪[6,0]+21𝒪[6,4]) + c₆*(-9𝒪[6,-6]-24𝒪[6,-5]-5𝒪[6,-2]-8𝒪[6,-1]+8𝒪[6,1]+24𝒪[6,5])


### Langevin

In [142]:
kT      = 1.7  * Sunny.meV_per_K         
tol     = 1e-2                 
nsteps  = 200

dt      =  0.09048
damping = 0.1     


sys_i = reshape_supercell(sys_lr, [4 0 0; 0 4 0; 0 0 4]) 
randomize_spins!(sys_i)

fig2 = Figure(size=(1000, 500))

langevin = Langevin(dt; damping, kT)
suggest_timestep(sys_i, langevin; tol=tol)

energies = [energy_per_site(sys_i)]
for _ in 1:nsteps
    step!(sys_i, langevin)
    push!(energies, energy_per_site(sys_i))
end
ax = Axis(fig2[1,1];
    xlabel = "Timesteps",
    ylabel = "Energy (meV)",
    title  = "dt = $(dt), damping = $(damping), tol = $(tol)")
lines!(ax, energies, color=:red)


fig2


Consider dt ≈ 0.09048 for this spin configuration at tol = 0.01000. Current value is dt = 0.09048.


In [ ]:
#Sample
# kT      = 1.7*Sunny.meV_per_K
# damping = 0.1
si=10 # q-space pixel size as 1/si

langevin = Langevin(dt; damping, kT)

# Energy range: max resolvable frequency ~ π/dt, but stay physically reasonable

energies = range(0, 5, 100)

sys_i = reshape_supercell(sys_lr, [si 0 0; 0 si 0; 0 0 si]) 
randomize_spins!(sys_i)

formfactors = [1 => FormFactor("Ho3")]
sc = SampledCorrelations(sys_i; dt, energies, measure=ssf_perp(sys_i; formfactors))

n_decorr  = nsteps   # steps between samples (matching your thermalization)
n_samples = 50     # more samples = smoother spectrum

@showprogress for sample in 1:n_samples #for sample in 1:n_samples
    for _ in 1:n_decorr
        step!(sys_i, langevin)
    end
    add_sample!(sc, sys_i)
end

Progress:  86%|███████████████████████████████████▎     |  ETA: 0:01:00

In [139]:
#Plot (Q,E)-colormap (energies are chosen, q's aren't yet)
fig = Figure(size=(800, 600))
qbs = 0.0


qbs  = 0.0:0.5:1.5
dirs = ["h", "h", "h", "h"]  # adjust per slice
npts = 200

fig = Figure(size=(1000, 700))

for (i, qb) in enumerate(qbs)
    path = [[-4, -4, qb], [4, 4, qb]]
    qpts = q_space_path(cryst, path, npts)
    Sqw  = intensities(sc, qpts; energies=:available, kT)

    xticks, xlabels = qticks_lang(5, path, npts, dirs[i])
    emax = energies[end]

    plot_intensities!(fig[fldmod1(i, 2)...], Sqw;
        title = "L = $qb",
        colormap = :viridis,
        colorrange = (0, 800),
        axis = (
            yticks      = 0:emax/emax:emax,
            xticks      = (xticks, xlabels),
            xlabel      = "(H H $(qb)) (r.l.u.)",
            ylabel      = "Energy (meV)"
        )
    )
end

Label(fig[0, 1:2], "Jnn = $(round(J_nn, digits=3)) meV, Ising-SIA = $(round(D, digits=3)) meV, NO dipole-dipole \nSystem size: $(si)x$(si)x$(si), samples = $(n_samples), tol: $(tol), damping: $(damping), dt: $(dt), T = $(round(kT/Sunny.meV_per_K, digits=3)) K"; fontsize=15, font=:bold)

fig

In [132]:
#Plot integrated over all E (Q,Q)-cut (Static intensities)
qpts = q_space_grid(cryst, [1, 1, 0], range(-6, 6, 200), [0, 0, 1], (-6, 6))
Sq  = intensities_static(sc, qpts; kT)

xticks, xlabels = qticks(11, [[-5, -5, 0], [5, 5, 0]], -5, 5, 'h')
yticks, ylabels = qticks(11, [[0, 0, -5], [0, 0, 5]], -5, 5, 'l')


fig = Figure(; size=(900,700))
plot_intensities!(fig[1,1], Sq; title="Jnn = $(round(J_nn, digits=3)) meV, Ising-SIA = $(round(D, digits=3)) meV, NO dipole-dipole \nSystem size: $(si)x$(si)x$(si), samples = $(n_samples), tol: $(tol), damping: $(damping), dt: $(dt), T = $(round(kT/Sunny.meV_per_K, digits=3)) K", 
    colormap=:viridis, colorrange=(0,4000), axis = (xticks = xticks, xlabel = "(H H 0)", yticks = yticks, ylabel= "(0 0 L)"))
ax = current_axis()
ax.aspect = DataAspect()
fig